In [1]:
%useLatestDescriptors

In [2]:
%use datetime
%use dataframe
%use kandy
%use ktor-client

In [17]:
import kotlin.time.Clock
import kotlinx.datetime.format.byUnicodePattern
import kotlinx.datetime.format.FormatStringsInDatetimeFormats
import kotlinx.datetime.format.byUnicodePattern
import java.net.URLEncoder
import java.nio.charset.StandardCharsets

val now = Clock.System.now()

@OptIn(FormatStringsInDatetimeFormats::class)
val currentTime = now
    .toLocalDateTime(TimeZone.of("Asia/Seoul"))
    .format(LocalDateTime.Format{byUnicodePattern("yyyy-MM-dd HH:mm:ss")})

@OptIn(FormatStringsInDatetimeFormats::class)
val previous24Hour = now
    .minus(1, DateTimeUnit.HOUR)
    .toLocalDateTime(TimeZone.of("Asia/Seoul"))
    .format(LocalDateTime.Format{byUnicodePattern("yyyy-MM-dd")})

print("Current time : ${currentTime}, Previous time : ${previous24Hour}")

val serviceKeyFilePath = "/Users/unchil/AndroidStudioProjects/OceanWaterInfo/collectionServer/src/main/resources/application.json"

val configData = DataRow.readJson(path=serviceKeyFilePath)

Current time : 2026-07-20 18:18:57, Previous time : 2026-07-20

In [59]:

val url = "${configData.MOF_API?.endPoint}/${configData.MOF_API?.subPath}" +
        "?wtch_dt_start=${URLEncoder.encode(previous24Hour, StandardCharsets.UTF_8.toString())}" +
        "&wtch_dt_end=${URLEncoder.encode(currentTime, StandardCharsets.UTF_8.toString())}" +
        "&numOfRows=1000" +
        "&ServiceKey=${configData.MOF_API?.apikey}"


In [60]:
@file:DependsOn("org.json:json:20250107")

In [61]:
import io.ktor.client.HttpClient
import io.ktor.client.engine.ProxyBuilder.http
import io.ktor.client.engine.cio.CIO
import io.ktor.client.plugins.HttpTimeout
import io.ktor.client.plugins.contentnegotiation.ContentNegotiation
import io.ktor.client.request.get
import io.ktor.client.statement.bodyAsText
import io.ktor.serialization.kotlinx.json.json
import org.json.XML
import kotlin.coroutines.suspendCoroutine


In [16]:
@Serializable
@SerialName("item")
data class OceanWaterQuality (
    val num: Int, // 순번
    val rtmWqWtchStaCd: Double, // 실시간수질관측정점코드
    val rtmWqWtchDtlDt: String, // 실시간수질관측상세일시
    val rtmWtchWtem: Double, // 실시간관측수온
    val rtmWqCndctv: Double, // 실시간수질전기전도도
    val ph: Double, // 수소이온농도
    val rtmWqDoxn: Double, // 실시간수질용존산소량
    val rtmWqTu: Double, // 실시간수질탁도
    val rtmWqBgalgsQy: Double?, // 실시간수질남조류량
    val rtmWqChpla: Double, // 실시간수질클로로필
    val rtmWqSlnty: Double // 실시간수질염분
)



In [62]:
fun loadData(path:String):DataFrame<*> {
    var requestPage = 1
    val rows = mutableListOf<DataFrame<*>>()

    do{
        val pagePath = "$path&pageNo=$requestPage"
        try {
            val response = http.get(pagePath)

            if (response.status.value == 200) {
                try {
                    XML.toJSONObject(response.bodyAsText()).let { jsonData ->
                        val df = DataFrame.readJson(jsonData.toString().byteInputStream())
                        val result = df.get("response").get("body").get("items").get("item")[0] as DataFrame<*>
                        requestPage += 1
                        rows.add(result)
                    }
                } catch(e: Exception) {
                    print(e.localizedMessage)
                    break
                }

            } else {
                println("${response.status.description}")
            }

        } catch(e:Exception ){
            requestPage += 1
            println(e.localizedMessage)

        }

    } while (requestPage < 500 )

    return rows.concat()

}


In [63]:
val dfRaw = loadData(url)

Column not found: 'items'

In [64]:
dfRaw.columnNames()

[rtmWqDoxn, rtmWqChpla, rtmWqBgalgsQy, rtmWqWtchStaCd, num, rtmWqTu, ph, rtmWqSlnty, rtmWqCndctv, rtmWqWtchDtlDt, rtmWtchWtem]

In [65]:
dfRaw.schema()

rtmWqDoxn: Double
rtmWqChpla: Comparable<*>
rtmWqBgalgsQy: String
rtmWqWtchStaCd: String
num: Int
rtmWqTu: Int
ph: Double
rtmWqSlnty: Double
rtmWqCndctv: Number
rtmWqWtchDtlDt: String
rtmWtchWtem: Double

In [66]:
val df = dfRaw.dropNA{ rtmWqBgalgsQy }.convert { rtmWqChpla }.with {
    val value = it?.toString()?.trim()
    if (value.isNullOrBlank()) 0.0 else value.toDouble()
}
df

rtmWqDoxn,rtmWqChpla,rtmWqBgalgsQy,rtmWqWtchStaCd,num,rtmWqTu,ph,rtmWqSlnty,rtmWqCndctv,rtmWqWtchDtlDt,rtmWtchWtem
6.610000,21.660000,,NEP2002,1,8,7.980000,17.900000,29.200001,2026-07-20 00:00:00.0,25.320000
4.264000,8.944000,,SEA2005,2,1,7.640000,29.424999,45.451000,2026-07-20 00:00:00.0,28.010000
5.500000,6.890000,,SEA5003,3,12,7.620000,32.582001,48.202000,2026-07-20 00:00:00.0,23.370001
5.430000,10.680000,,SEA6001,4,43,7.810000,31.197001,47.959000,2026-07-20 00:00:00.0,26.950001
2.892000,1.457000,,NEP3001,5,9,7.420000,13.396000,22.264999,2026-07-20 00:00:00.0,25.920000
4.057000,2.041000,,NEP2001,6,14,7.140000,1.276000,2.478000,2026-07-20 00:00:00.0,27.230000
2.720000,0.000000,,SEA1005,7,2,7.520000,5.306000,9.493000,2026-07-20 00:00:00.0,27.100000
4.189000,1.307000,,SEA2007,8,31,7.920000,31.983999,51.693001,2026-07-20 00:00:00.0,25.900000
4.004000,7.251000,,SEA2005,9,1,7.640000,29.737000,45.883999,2026-07-20 00:10:00.0,27.850000
5.870000,7.320000,,SEA5003,10,11,7.630000,32.608002,48.191002,2026-07-20 00:10:00.0,23.330000


In [68]:
df.schema()

rtmWqDoxn: Double
rtmWqChpla: Double
rtmWqBgalgsQy: String
rtmWqWtchStaCd: String
num: Int
rtmWqTu: Int
ph: Double
rtmWqSlnty: Double
rtmWqCndctv: Number
rtmWqWtchDtlDt: String
rtmWtchWtem: Double

In [69]:
val renamedDf = df.rename(
    "rtmWqDoxn" to "용존산소",
    "rtmWqChpla" to "클로로필",
    "rtmWqWtchStaCd" to "관측정점코드",
    "num" to "순번",
    "rtmWqTu" to "탁도",
    "ph" to "수소이온농도",
    "rtmWqSlnty" to "염분",
    "rtmWqCndctv" to "전기전도도",
    "rtmWqWtchDtlDt" to "일시",
    "rtmWtchWtem" to "수온"
)
renamedDf

용존산소,클로로필,rtmWqBgalgsQy,관측정점코드,순번,탁도,수소이온농도,염분,전기전도도,일시,수온
6.610000,21.660000,,NEP2002,1,8,7.980000,17.900000,29.200001,2026-07-20 00:00:00.0,25.320000
4.264000,8.944000,,SEA2005,2,1,7.640000,29.424999,45.451000,2026-07-20 00:00:00.0,28.010000
5.500000,6.890000,,SEA5003,3,12,7.620000,32.582001,48.202000,2026-07-20 00:00:00.0,23.370001
5.430000,10.680000,,SEA6001,4,43,7.810000,31.197001,47.959000,2026-07-20 00:00:00.0,26.950001
2.892000,1.457000,,NEP3001,5,9,7.420000,13.396000,22.264999,2026-07-20 00:00:00.0,25.920000
4.057000,2.041000,,NEP2001,6,14,7.140000,1.276000,2.478000,2026-07-20 00:00:00.0,27.230000
2.720000,0.000000,,SEA1005,7,2,7.520000,5.306000,9.493000,2026-07-20 00:00:00.0,27.100000
4.189000,1.307000,,SEA2007,8,31,7.920000,31.983999,51.693001,2026-07-20 00:00:00.0,25.900000
4.004000,7.251000,,SEA2005,9,1,7.640000,29.737000,45.883999,2026-07-20 00:10:00.0,27.850000
5.870000,7.320000,,SEA5003,10,11,7.630000,32.608002,48.191002,2026-07-20 00:10:00.0,23.330000


In [70]:
renamedDf.schema()

용존산소: Double
클로로필: Double
rtmWqBgalgsQy: String
관측정점코드: String
순번: Int
탁도: Int
수소이온농도: Double
염분: Double
전기전도도: Number
일시: String
수온: Double